In [1]:
# Import the packages we will use
#Utilities
import os
import re
import itertools
import glob
import pickle
import argparse

#Data Management
import numpy as np
from numpy import diff
import pandas as pd
import h5py
import scipy
from scipy.stats import linregress
from scipy import ndimage
from functools import partial
from scipy.linalg import toeplitz

#Plotting
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib import cm
from matplotlib.gridspec import GridSpec
from matplotlib.gridspec import GridSpecFromSubplotSpec
import matplotlib.colors as colors
from matplotlib.colors import ListedColormap
import seaborn as sns
import upsetplot
from upsetplot import UpSet

#Genomics
import pairtools
import cooler
import cooltools
import bioframe
from bioframe import overlap

/home/xiangru.huo-umw/conda/Jupiter/lib/python3.11/site-packages/cooltools/lib/numutils.py:651: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit  # (nopython=True)
/home/xiangru.huo-umw/conda/Jupiter/lib/python3.11/site-packages/cooltools/lib/numutils.py:726: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit  # (nopython=True)


In [7]:
dataDir = '/home/xiangru.huo-umw/scHiC/benchmark/scHiC_1203_2022/1+2_sc_results'#'dir/to/mapped/mcools'
outDataDir = '/home/xiangru.huo-umw/scHiC/benchmark/plot_files'

conditions = [
    'G1_merge_1203',
    'proM_merge_1203'
]

long_names = {
    'G1_merge_1203' : 'G1_merge',
    'proM_merge_1203' : 'proM_merge'
}

In [8]:
#Read in coolers - 10kb bins
binsize = 10000

clr_paths_10kb = {}
for cond in conditions:
    clr_paths_10kb[cond] = f'{dataDir}/coolers_library/{long_names[cond]}.hg38.mapq_30.1000.mcool::resolutions/{binsize}'
        
clrs10kb = {
    cond: cooler.Cooler(clr_paths_10kb[cond]) for cond in conditions
}

In [2]:
# Use bioframe to fetch the genomic features from the UCSC.
hg38_chromsizes_all = bioframe.fetch_chromsizes('hg38')
hg38_cens = bioframe.fetch_centromeres('hg38')
hg38_arms = bioframe.core.construction.add_ucsc_name_column(bioframe.make_chromarms(hg38_chromsizes_all,  hg38_cens))

In [3]:
print(bioframe.__version__)

0.4.1


In [4]:
print(cooltools.__version__)

0.5.4


In [5]:
print(cooler.__version__)

0.9.2


In [9]:
#calculate read depth?
clrs10kb['G1_merge_1203'].info #Sum is the same as read # actually used in making cooler

{'bin-size': 10000,
 'bin-type': 'fixed',
 'creation-date': '2023-01-10T19:36:29.094827',
 'format': 'HDF5::Cooler',
 'format-url': 'https://github.com/mirnylab/cooler',
 'format-version': 3,
 'generated-by': 'cooler-0.8.5',
 'genome-assembly': 'unknown',
 'metadata': {},
 'nbins': 308839,
 'nchroms': 25,
 'nnz': 871776,
 'storage-mode': 'symmetric-upper',
 'sum': 4012711}

In [10]:
#check another file info
clrs10kb['proM_merge_1203'].info

{'bin-size': 10000,
 'bin-type': 'fixed',
 'creation-date': '2023-01-10T20:24:45.178187',
 'format': 'HDF5::Cooler',
 'format-url': 'https://github.com/mirnylab/cooler',
 'format-version': 3,
 'generated-by': 'cooler-0.8.5',
 'genome-assembly': 'unknown',
 'metadata': {},
 'nbins': 308839,
 'nchroms': 25,
 'nnz': 1811473,
 'storage-mode': 'symmetric-upper',
 'sum': 8665756}

In [11]:
#Sample for just 1000bp cooler (smallest) - then combine replicates and zoomify
sample_binsize = 1000

In [12]:
clrPaths = {}
for cond in conditions:
    clrPaths[cond] = f'{dataDir}/coolers_library/{long_names[cond]}.hg38.mapq_30.1000.mcool::resolutions/{sample_binsize}'

In [20]:
import os
os.getcwd()

'/home/xiangru.huo-umw/scHiC/scripts'

In [23]:
#sample each to approx 4012711 in this case - has to be one less than the sum of the smallest cooler
#We use the bsub job scheduler - this can be modified based on your computing resources

for cond in conditions:
    in_fname = clrPaths[cond]
    out_fname = f'{outDataDir}/cooler_data/{long_names[cond]}.1203.hg38.mapq_30.1000.cool'
    !bsub -q short -W 01:00 -e ./dis.err -o ./dis.log \
        -n 1 -R "rusage[mem=50000]" -R "span[hosts=1]" \
        "cooltools random-sample -c 4012000 --exact $in_fname $out_fname"

Job <737316> is submitted to queue <short>.
Job <737317> is submitted to queue <short>.


In [41]:
sampledPaths = {}
for cond in conditions:
    sampledPaths[cond] = f'{outDataDir}/cooler_data/{long_names[cond]}.1203.hg38.mapq_30.1000.cool'

In [29]:
#Zoomify and balance sampled cooler files
for cond in conditions:
    coolfile = sampledPaths[cond]
    mcoolfile = f'{outDataDir}/cooler_data/{long_names[cond]}.1203.hg38.mapq_30.1000.mcool'
    !bsub -q short -W 01:00 -e ./dis.err -o ./dis.log \
        -n 1 -R "rusage[mem=50000]" -R "span[hosts=1]" \
        "cooler zoomify -p 4 --balance --resolutions 4DN $coolfile -o $mcoolfile"

Job <737332> is submitted to queue <short>.
Job <737333> is submitted to queue <short>.


In [44]:
#batch2 data
dataDir = '/home/xiangru.huo-umw/scHiC/benchmark/scHiC_0115_2023/results_12merge'#'dir/to/mapped/mcools'
outDataDir = '/home/xiangru.huo-umw/scHiC/benchmark/plot_files'

conditions_2 = [
    'G1_merge_0115',
    'proM_merge_0115'
]

long_names_2 = {
    'G1_merge_0115' : 'G1_merge',
    'proM_merge_0115' : 'proM_merge'
}

#Read in coolers - 10kb bins
binsize = 10000

clr_paths_10kb = {}
for cond in conditions_2:
    clr_paths_10kb[cond] = f'{dataDir}/coolers_library/{long_names_2[cond]}.hg38.mapq_30.1000.mcool::resolutions/{binsize}'
        
clrs10kb = {
    cond: cooler.Cooler(clr_paths_10kb[cond]) for cond in conditions_2
}

In [45]:
#calculate read depth?
clrs10kb['G1_merge_0115'].info #Sum is the same as read # actually used in making cooler

{'bin-size': 10000,
 'bin-type': 'fixed',
 'creation-date': '2023-03-02T18:43:11.388135',
 'format': 'HDF5::Cooler',
 'format-url': 'https://github.com/mirnylab/cooler',
 'format-version': 3,
 'generated-by': 'cooler-0.8.5',
 'genome-assembly': 'unknown',
 'metadata': {},
 'nbins': 308839,
 'nchroms': 25,
 'nnz': 1048870,
 'storage-mode': 'symmetric-upper',
 'sum': 5223861}

In [46]:
#check another file info
clrs10kb['proM_merge_0115'].info

{'bin-size': 10000,
 'bin-type': 'fixed',
 'creation-date': '2023-03-02T18:46:39.625709',
 'format': 'HDF5::Cooler',
 'format-url': 'https://github.com/mirnylab/cooler',
 'format-version': 3,
 'generated-by': 'cooler-0.8.5',
 'genome-assembly': 'unknown',
 'metadata': {},
 'nbins': 308839,
 'nchroms': 25,
 'nnz': 786337,
 'storage-mode': 'symmetric-upper',
 'sum': 3448902}

In [47]:
#Sample for just 1000bp cooler (smallest) - then combine replicates and zoomify
sample_binsize = 1000

clrPaths = {}
for cond in conditions_2:
    clrPaths[cond] = f'{dataDir}/coolers_library/{long_names_2[cond]}.hg38.mapq_30.1000.mcool::resolutions/{sample_binsize}'

In [48]:
#sample each to approx 3448902 in this case - has to be one less than the sum of the smallest cooler
#We use the bsub job scheduler - this can be modified based on your computing resources

for cond in conditions_2:
    in_fname = clrPaths[cond]
    out_fname = f'{outDataDir}/cooler_data/{long_names_2[cond]}.0115.hg38.mapq_30.1000.cool'
    !bsub -q short -W 01:00 -e ./dis.err -o ./dis.log \
        -n 1 -R "rusage[mem=50000]" -R "span[hosts=1]" \
        "cooltools random-sample -c 3448000 --exact $in_fname $out_fname"

Job <737352> is submitted to queue <short>.
Job <737353> is submitted to queue <short>.


In [49]:
sampledPaths_2 = {}
for cond in conditions_2:
    sampledPaths_2[cond] = f'{outDataDir}/cooler_data/{long_names_2[cond]}.0115.hg38.mapq_30.1000.cool'

In [43]:
#Zoomify and balance sampled cooler files
for cond in conditions_2:
    coolfile = sampledPaths_2[cond]
    mcoolfile = f'{outDataDir}/cooler_data/{long_names_2[cond]}.0115.hg38.mapq_30.1000.mcool'
    !bsub -q short -W 01:00 -e ./dis.err -o ./dis.log \
        -n 1 -R "rusage[mem=50000]" -R "span[hosts=1]" \
        "cooler zoomify -p 4 --balance --resolutions 4DN $coolfile -o $mcoolfile"

Job <737350> is submitted to queue <short>.
Job <737351> is submitted to queue <short>.


In [52]:
ComboConds = [
    'G1_merge_2combo',
    'proM_merge_2combo'
]

long_names_combo = {
    'G1_merge_2combo' : 'G1_merge',
    'proM_merge_2combo' : 'proM_merge'
}

sampledPaths_combo = {}
for cond in ComboConds:
    sampledPaths_combo[cond] = f'{outDataDir}/cooler_data/{long_names_combo[cond]}.2combo.hg38.mapq_30.1000.cool'

In [53]:
#Merge replicates for combined files
for (cond1, cond2, combinedCond) in zip(conditions, conditions_2, ComboConds):
    
    cond1Sampled = sampledPaths[cond1]
    cond2Sampled = sampledPaths_2[cond2]

    combinedfile = sampledPaths_combo[combinedCond]
    
    cooler.merge_coolers(combinedfile, [cond1Sampled, cond2Sampled], 20000000)